# Task 4: style-conditioned face-to-sketch GAN (Kaggle run)

Same settings as before: GPU on, Internet on, `WANDB_API_KEY` secret attached
(and `GITHUB_TOKEN` if the repository is private).

FS2K is hosted on Google Drive. Cell 3 first looks for it in an attached Kaggle dataset
(recommended: upload the FS2K zip as a private Kaggle dataset and attach it with Add Input),
and otherwise tries to download it from Google Drive with `gdown`.

In [ ]:
# 1. Secrets -> environment variables
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "genai-a1"

REPO = "aleenababar04/genai-assignment-1"
try:
    token = secrets.get_secret("GITHUB_TOKEN")   # only needed for a private repo
    REPO_URL = f"https://{token}@github.com/{REPO}.git"
except Exception:
    REPO_URL = f"https://github.com/{REPO}.git"


In [ ]:
# 2. Get the code and install the few packages Kaggle does not ship
%cd /kaggle/working
!rm -rf genai-assignment-1
!git clone -q $REPO_URL genai-assignment-1
%cd genai-assignment-1
!git log --oneline -1
!pip install -q optuna wandb onnx onnxruntime pytorch-msssim python-dotenv gdown

import torch
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Turn the GPU on in the notebook settings."


In [ ]:
# 3. Put FS2K at data/FS2K (from an attached Kaggle dataset, or from Google Drive)
import glob, shutil, subprocess, zipfile

os.makedirs("data", exist_ok=True)
found = glob.glob("/kaggle/input/**/anno_train.json", recursive=True)
if found:
    source = os.path.dirname(found[0])
    print("using attached dataset:", source)
    shutil.copytree(source, "data/FS2K", dirs_exist_ok=True)
else:
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if not zips:
        # File id from the official FS2K repository's Google Drive link.
        subprocess.run(["gdown", "1saIMhQ3dc5_ftkfGmBPbCluRn_zy7QQp", "-O", "data/FS2K.zip"], check=True)
        zips = ["data/FS2K.zip"]
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("data/FS2K")
!ls data/FS2K


In [ ]:
# 4. Official train/test split + stratified 15% validation, 128x128 cache
#    (the split is checked against manifests/fs2k_split.json if it is in the repo)
!python -m src.data.prepare_fs2k
!python -m pytest -q


In [ ]:
# 5. Optuna search with shorter runs, then the full schedule -> checkpoints/task4_cgan.pt
!python -m src.training.train_cgan --mode optuna
!python -m src.training.train_cgan --mode final


In [ ]:
# 6. Test evaluation and ONNX export of the generator
!python -m src.evaluation.eval_cgan
!python -m src.export.export_onnx --task task4


In [ ]:
# 7. Collect everything that must go back into the repository.
#    Download task4_outputs.zip from the notebook's Output panel.
!mkdir -p /kaggle/working/task4_outputs/configs /kaggle/working/task4_outputs/manifests /kaggle/working/task4_outputs/checkpoints
!cp configs/task4_cgan_best.yaml /kaggle/working/task4_outputs/configs/
!cp manifests/fs2k_split.json /kaggle/working/task4_outputs/manifests/
!cp checkpoints/task4_cgan.pt /kaggle/working/task4_outputs/checkpoints/
!cp -r optuna_studies models report/results report/figures /kaggle/working/task4_outputs/
%cd /kaggle/working
!zip -qr task4_outputs.zip task4_outputs
!ls -lh task4_outputs.zip
